# Riemannian pipeline на `new_format_data`

- Окна: **1 с**, 18 каналов (`raw_signals` → `18×200`)
- Метка: `has_depd`
- Split: **80/20 по пациентам** (`external_id`)
- Модель: `Covariances` → `TangentSpace` → `LogisticRegression`

CLI из `diploma_MAG`:
```bash
python -m riemann_new_format --data-dir new_format_data --reuse-split
```

In [ ]:
import sys
from pathlib import Path

ROOT = Path("/Users/vspyatochkin/diploma_MAG")
sys.path.insert(0, str(ROOT))

from riemann_new_format.dataset import (
    NewFormatDataset,
    patient_train_test_split,
    indices_for_patients,
    labels_for_indices,
    subsample_train_indices,
    save_split,
)
from riemann_new_format.pipeline import build_riemann_pipeline, evaluate_binary, predict_in_batches

DATA_DIR = ROOT / "new_format_data"

In [ ]:
ds = NewFormatDataset(DATA_DIR)
train_patients, test_patients = patient_train_test_split(ds.meta, test_size=0.2, random_state=42)
save_split(DATA_DIR, train_patients, test_patients)

assert not set(train_patients) & set(test_patients)
print(f"Пациентов train/test: {len(train_patients)} / {len(test_patients)}")

train_idx = indices_for_patients(ds.meta, train_patients)
test_idx = indices_for_patients(ds.meta, test_patients)
y_train_full = labels_for_indices(ds.meta, train_idx)
y_test = labels_for_indices(ds.meta, test_idx)

train_idx_fit = subsample_train_indices(train_idx, y_train_full, neg_per_pos=5.0, max_samples=200_000)
y_train = labels_for_indices(ds.meta, train_idx_fit)
print(f"Train fit: {len(train_idx_fit):,} | DEPD {y_train.sum():,}")
print(f"Test:      {len(test_idx):,} | DEPD {y_test.sum():,}")

In [ ]:
X_train = ds.get_batch(train_idx_fit)
model = build_riemann_pipeline(classifier="lr", cov_estimator="lwf")
model.fit(X_train, y_train)

y_pred = predict_in_batches(model, ds._raw, test_idx, batch_size=512)
metrics = evaluate_binary(y_test, y_pred)
for k in ("f1", "precision", "recall", "specificity", "accuracy"):
    print(f"{k:14}: {metrics[k]:.4f}")